In [5]:
import apache_beam as beam
import json
import importlib
import os
import unittest

In [6]:
fromNotebook = True
configFile = "warehouse_wwi.json"
loadConfigFile = "load_wwi.json"
newCutoffDate = "2013-01-01"

In [7]:
prefix = ""
if fromNotebook == False:
    prefix = "notebooks/"

file_path = os.path.join(os.getcwd(), f"{prefix}modules", 'sql_utilities.py')
spec = importlib.util.spec_from_file_location("sql_utils", file_path)
sql_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sql_utils)

In [8]:
f = open(loadConfigFile,)
source_config = json.load(f)
f.close()

load_tables = [
    {
        "name": table["name"],
        "schema": table["destination"]["schema"],
        "table": table["destination"]["table"]
    }
    for table in source_config["tables"]
]

print(f"load_tables: {load_tables}")

load_tables: [{'name': 'ApplicationCities', 'schema': 'dbo', 'table': 'Application_Cities'}, {'name': 'ApplicationCitiesArchive', 'schema': 'dbo', 'table': 'Application_Cities_Archive'}, {'name': 'ApplicationCountries', 'schema': 'dbo', 'table': 'Application_Countries'}, {'name': 'ApplicationCountriesArchive', 'schema': 'dbo', 'table': 'Application_Countries_Archive'}, {'name': 'ApplicationDeliveryMethods', 'schema': 'dbo', 'table': 'Application_DeliveryMethods'}, {'name': 'ApplicationDeliveryMethodsArchive', 'schema': 'dbo', 'table': 'Application_DeliveryMethods_Archive'}, {'name': 'ApplicationPaymentMethods', 'schema': 'dbo', 'table': 'Application_PaymentMethods'}, {'name': 'ApplicationPaymentMethodsArchive', 'schema': 'dbo', 'table': 'Application_PaymentMethods_Archive'}, {'name': 'ApplicationPeople', 'schema': 'dbo', 'table': 'Application_People'}, {'name': 'ApplicationPeopleArchive', 'schema': 'dbo', 'table': 'Application_People_Archive'}, {'name': 'ApplicationStateProvinces', 'sc

In [9]:
f = open(configFile,)
config = json.load(f)
f.close()

source_db = config["source"]
destination_db = config["destination"]
tables = config["tables"]
initialCutoffDate = config["initialCutoffDate"]
if fromNotebook == True:
    newCutoffDate = config["cutoffDate"]
lastCutoffDate = "2012-12-31"
warehouse_tables = [
    {
        "name": table["name"],
        "schema": table["schema"],
        "table": table["table"]
    }
    for gTables in tables for table in gTables 
]

print(f"source_db: {source_db}")
print(f"destination_db: {destination_db}")
print(f"tables: {tables}")
print(f"initialCutoffDate: {initialCutoffDate}")
print(f"newCutoffDate: {newCutoffDate}")
print(f"lastCutoffDate: {lastCutoffDate}")
print(f"warehouse_tables: {warehouse_tables}")

source_db: {'databaseType': 'mssql', 'database': 'WideWorldImportersDW', 'conn': 'DRIVER={ODBC Driver 17 for SQL Server};SERVER=localhost,1433;DATABASE=WideWorldImportersDW;UID=sa;PWD=PP@$$w0rd'}
destination_db: {'databaseType': 'mssql', 'database': 'WideWorldImportersDW', 'conn': 'DRIVER={ODBC Driver 17 for SQL Server};SERVER=localhost,1433;DATABASE=WideWorldImportersDW;UID=sa;PWD=PP@$$w0rd'}
tables: [[{'name': 'DimCities', 'schema': 'dbo', 'table': 'DimCities', 'createTable': 'scripts/warehouse_wwi_mssql/create_dimension_cities.sql', 'upsertTable': 'scripts/warehouse_wwi_mssql/upsert_dimension_cities_2013-01-01.sql', 'specificColumnTypes': [{'name': 'Location', 'type': 'VARBINARY(MAX)', 'typeCast': 'CAST(? AS VARBINARY(MAX))'}], 'primaryKey': 'CityKey', 'modifyTable': [], 'unitTests': [{'name': 'Test CityKey Not Null', 'type': 'column', 'column': 'CityKey', 'subType': 'notNull'}, {'name': 'Test CityKey Unique', 'type': 'column', 'column': 'CityKey', 'subType': 'unique'}, {'name': 'Te

In [10]:
class UpsertData(beam.DoFn):
    def __init__(self, database_type, conn_str, upsert_path, values_to_replace, tables_to_replace, yield_record = None, show_sql = False):
        self.database_type = database_type
        self.conn_str = conn_str
        self.upsert_path = upsert_path
        self.values_to_replace = values_to_replace
        self.tables_to_replace = tables_to_replace
        self.yield_record = yield_record
        self.show_sql = show_sql

    def process(self, element):
        sql = sql_utils.get_sql_from_script(
            self.upsert_path, 
            self.values_to_replace, 
            self.tables_to_replace
        )

        if self.show_sql == True:
            print(f"""Upsert Data sql:
                  {sql} 
                  """)
        
        sql_utils.exec_sql(self.conn_str, sql, None, self.database_type)

        if self.yield_record is not None:
            yield self.yield_record

In [11]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
tc = unittest.TestCase()

class AssertData(beam.DoFn):
    def __init__(self, database_type, conn_str, test_config, unit_tests, values_to_replace, tables_to_replace, test, test_record_count, yield_record = None, print_sql = False):
        self.database_type = database_type
        self.conn_str = conn_str
        self.test_config = test_config
        self.unit_tests = unit_tests
        self.values_to_replace = values_to_replace
        self.tables_to_replace = tables_to_replace
        self.test = test
        self.test_record_count = test_record_count
        self.yield_record = yield_record
        self.print_sql = print_sql

    def process(self, element):
        if self.test == True:
            error_counts = []

            if len(self.unit_tests) > 0:
                count_sqls = []
                
                for unit_test in self.unit_tests:
                    common_values = [
                        { "name": "Name", "value": unit_test["name"] },
                        { "name": "Column", "value": unit_test["column"] },
                        { "name": "Type", "value": unit_test["type"] },
                        { "name": "SubType", "value": unit_test["subType"] }
                    ]

                    if unit_test["type"] == "column":
                        if unit_test["subType"] == "relationship":
                            count_sqls.append(
                                sql_utils.get_sql_from_script(
                                    self.test_config[unit_test["subType"]],
                                    self.values_to_replace + common_values + [
                                        { "name": "ParentColumn", "value": unit_test["parentColumn"] },
                                        { "name": "ParentTable", "value": unit_test["parentTable"] }
                                    ], 
                                    self.tables_to_replace
                                ) 
                            )
                        elif unit_test["subType"] == "expressionTrue":
                            count_sqls.append(
                                sql_utils.get_sql_from_script(
                                    self.test_config[unit_test["subType"]],
                                    self.values_to_replace + common_values + [
                                        { "name": "Expression", "value": unit_test["expression"] }
                                    ], 
                                    self.tables_to_replace
                                ) 
                            )
                        else:
                            count_sqls.append(
                                sql_utils.get_sql_from_script(
                                    self.test_config[unit_test["subType"]],
                                    self.values_to_replace + common_values, 
                                    self.tables_to_replace
                                ) 
                            )
                    elif unit_test["type"] == "table":
                        count_sqls.append(
                            sql_utils.get_sql_from_script(
                                unit_test["testScript"],
                                self.values_to_replace + common_values, 
                                self.tables_to_replace
                            ) 
                        )

                if len(count_sqls) > 0:
                    count_sql = sql_utils.get_sql_from_script(
                        self.test_config["selectCounts"],
                        [
                            {
                                "name": "SelectCountsSql",
                                "value": " UNION ALL ".join(count_sqls)
                            }
                        ],
                        self.tables_to_replace
                    )

                    if self.print_sql == True:
                        print(f"count_sql: {count_sql}",)

                    error_counts = list(
                        sql_utils.select_sql(
                            self.conn_str, 
                            count_sql, 
                            self.database_type
                        )
                    )

            if len(error_counts) > 0:
                print("Error occurred on the following tables:")
                
                display(pd.DataFrame(error_counts))

                raise("Assert data error.  Please see error above.")
        
        if self.yield_record is not None:
            yield self.yield_record

In [13]:
p = beam.Pipeline()

def get_last_cutoff_date(table_name):
    lastCutoffDateFromLoadHistory = sql_utils.exec_sql_scalar(
        destination_db["conn"],
        sql_utils.get_sql_from_script(
            config["warehouseHistory"]["loadLastCutoffDateWarehouseHistoryTable"], 
            [
                { "name": "TableName", "value": table_name },
                { "name": "Schema", "value": config["warehouseHistory"]["schema"] },
                { "name": "Table", "value": config["warehouseHistory"]["table"] }
            ]
        ),
        destination_db["databaseType"]
    )

    if lastCutoffDateFromLoadHistory is None:
        return initialCutoffDate
    else:
        return lastCutoffDateFromLoadHistory.strftime("%Y-%m-%d %H:%M:%S")

def get_date_values(last_catuoff_date, new_cutoff_date):
    return [
        { "name": "LastCutoffDate", "value": last_catuoff_date },
        { "name": "NewCutoffDate", "value": new_cutoff_date }
    ]

def modify_table(p, table):
    tableName = table["table"]
    tableLastCutoffDate = get_last_cutoff_date(tableName)
    modifyTableScripts = []
                
    for modifyTableScript in table["modifyTable"]:
        modifyTableScripts.append(
            sql_utils.get_sql_from_script(
                config["modifyWarehouseHistory"]["loadUnprocessedModifyWarehouseHistoryScriptTable"],
                [
                    { "name": "ScriptName", "value": modifyTableScript["tableUpdate"] },
                    { "name": "LoadScriptName", "value": modifyTableScript["tableDataLoad"] },
                    { "name": "UpdateScriptName", "value": modifyTableScript["tableDataUpdate"] },
                    { "name": "Name", "value": modifyTableScript["name"] }
                ]
            )
        )
    
    unprocessedScripts = sql_utils.select_sql(
        destination_db["conn"],
        sql_utils.get_sql_from_script(
            config["modifyWarehouseHistory"]["loadUnprocessedModifyWarehouseHistoryTable"],
            [
                { "name": "Schema", "value": config["modifyWarehouseHistory"]["schema"] },
                { "name": "Table", "value": config["modifyWarehouseHistory"]["table"] },
                { "name": "TableName", "value": tableName },
                { "name": "LoadDate", "value": lastCutoffDate },
                { "name": "ModifyTableScripts", "value": " UNION ".join(modifyTableScripts) }
            ]
        ),
        destination_db["databaseType"]
    )

    input = (
        p
        | f"Modify Create Empty collection for {tableName}" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }]) 
    )
    
    for script in unprocessedScripts:
        print(f"Processing '{script["Name"]}' for table {tableName} for cutoff {newCutoffDate}.")
        
        sql_utils.exec_sql(
            destination_db["conn"],
            sql_utils.get_sql_from_script(
                script["ScriptName"], 
                [
                    { "name": "Database", "value": destination_db["database"] },
                    { "name": "Schema", "value": table["schema"] },
                    { "name": "Table", "value": table["table"] },
                ]
            ),
            None,
            destination_db["databaseType"]
        )

        input = (
            input
            | f"Modify Table {tableName} Upsert Data" >> 
                beam.ParDo(
                    UpsertData(
                        destination_db["databaseType"],
                        destination_db["conn"],
                        script["UpdateScriptName"],
                        get_date_values(
                            tableLastCutoffDate, 
                            newCutoffDate
                        ),
                        load_tables + warehouse_tables,
                        { "PlaceHolder": "Temp 1 record" }
                    )
                )
            | f"Modify Table {tableName} Assert Data" >>
                beam.ParDo(
                    AssertData(
                        database_type=destination_db["databaseType"], 
                        conn_str=destination_db["conn"], 
                        test_config=config["testConfig"], 
                        unit_tests=modifyTableScript["unitTests"], 
                        values_to_replace=[
                            { "name": "Database", "value": destination_db["database"] },
                            { "name": "Schema", "value": table["schema"] },
                            { "name": "Table", "value": table["table"] },
                            { "name": "LoadDate", "value": lastCutoffDate },
                            { "name": "LastCutoffDate", "value": lastCutoffDate },
                            { "name": "NewCutoffDate", "value": newCutoffDate }
                        ], 
                        tables_to_replace=load_tables + warehouse_tables, 
                        test=config["test"], 
                        test_record_count=config["testRecordCount"], 
                        yield_record={ "PlaceHolder": "Temp 1 record" }, 
                        print_sql=False
                    )
                )
            | f"Create Modify Table History Record for {tableName}" >>
                beam.ParDo(
                    UpsertData(
                        destination_db["databaseType"],
                        destination_db["conn"],
                        config["modifyWarehouseHistory"]["insertModifyWarehouseHistoryTable"],
                        [
                            { "name": "Schema", "value": config["modifyWarehouseHistory"]["schema"] },
                            { "name": "Table", "value": config["modifyWarehouseHistory"]["table"] },
                            { "name": "TableName", "value": tableName },
                            { "name": "LoadDate", "value": lastCutoffDate },
                            { "name": "Status", "value": "Successful" },
                            { "name": "Details", "value": "" },
                            { "name": "ScriptName", "value": script["Name"] }
                        ],
                        [],
                        { "PlaceHolder": "Temp 1 record" }
                    )
                ) 
        )
    
    return input

# Create warehouse history table if not existing
(
    p
    | "Input Create Warehouse History Table" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])
    | "Create Warehouse History Table" >> 
        beam.ParDo(
            UpsertData(
                destination_db["databaseType"],
                destination_db["conn"],
                config["warehouseHistory"]["createWarehouseHistoryTable"],
                [
                    { "name": "Database", "value": config["destination"]["database"] },
                    { "name": "Schema", "value": config["warehouseHistory"]["schema"] },
                    { "name": "Table", "value": config["warehouseHistory"]["table"] }
                ],
                [],
                { "PlaceHolder": "Temp 1 record" }
            )
        )
    | "Create Modify Warehouse History Table" >>
        beam.ParDo(
            UpsertData(
                destination_db["databaseType"],
                destination_db["conn"],
                config["modifyWarehouseHistory"]["createModifyWarehouseHistoryTable"],
                [
                    { "name": "Database", "value": config["destination"]["database"] },
                    { "name": "Schema", "value": config["modifyWarehouseHistory"]["schema"] },
                    { "name": "Table", "value": config["modifyWarehouseHistory"]["table"] }
                ],
                [],
                { "PlaceHolder": "Temp 1 record" }
            )
        )
)    

p.run().wait_until_finish()

for gTables in tables:
    p = beam.Pipeline()
    
    for table in gTables: 
        tableName = table["name"]

        lastCutoffDate = get_last_cutoff_date(tableName)

        print(f"Processing table {tableName} for {newCutoffDate}.")
        
        if lastCutoffDate >= newCutoffDate:
            print(f"Table {tableName} is already processed for {newCutoffDate}.")
        else:

            input = p | f"Input Create Table for {tableName}" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])
            
            if len(table["modifyTable"]) > 0:
                input = modify_table(p, table)

            (
                input
                | f"Create Table for {tableName}" >>
                    beam.ParDo(
                        UpsertData(
                            destination_db["databaseType"],
                            destination_db["conn"],
                            table["createTable"],
                            [
                                { "name": "Database", "value": destination_db["database"] },
                                { "name": "Schema", "value": table["schema"] },
                                { "name": "Table", "value": table["table"] }
                            ],
                            [],
                            { "PlaceHolder": "Temp 1 record" }
                        )
                    )
                | f"Upsert Data for {tableName}" >> 
                    beam.ParDo(
                        UpsertData(
                            destination_db["databaseType"],
                            destination_db["conn"],
                            table["upsertTable"],
                            get_date_values(
                                lastCutoffDate, 
                                newCutoffDate
                            ),
                            load_tables + warehouse_tables,
                            { "PlaceHolder": "Temp 1 record" },
                            False
                        )
                    )
                | f"Assert Data for {tableName}" >>
                    beam.ParDo(
                        AssertData(
                            database_type=destination_db["databaseType"], 
                            conn_str=destination_db["conn"], 
                            test_config=config["testConfig"], 
                            unit_tests=table["unitTests"], 
                            values_to_replace=[
                                { "name": "Database", "value": destination_db["database"] },
                                { "name": "Schema", "value": table["schema"] },
                                { "name": "Table", "value": table["table"] },
                                { "name": "LoadDate", "value": newCutoffDate },
                                { "name": "LastCutoffDate", "value": lastCutoffDate },
                                { "name": "NewCutoffDate", "value": newCutoffDate }
                            ], 
                            tables_to_replace=load_tables + warehouse_tables, 
                            test=config["test"], 
                            test_record_count=config["testRecordCount"], 
                            yield_record={ "PlaceHolder": "Temp 1 record" }, 
                            print_sql=False
                        )
                    )
                | f"Insert Warehouse History Record Successful for {tableName}" >>
                    beam.ParDo(
                        UpsertData(
                            destination_db["databaseType"],
                            destination_db["conn"],
                            config["warehouseHistory"]["insertWarehouseHistoryTable"],
                            [
                                { "name": "Schema", "value": config["warehouseHistory"]["schema"] },
                                { "name": "Table", "value": config["warehouseHistory"]["table"] },
                                { "name": "TableName", "value": tableName },
                                { "name": "LoadDate", "value": newCutoffDate },
                                { "name": "Status", "value": "Successful" },
                                { "name": "Details", "value": "" }
                            ],
                            [],
                            { f"Successfully processed table {tableName} after {lastCutoffDate} to {newCutoffDate}." }
                        )
                    )
                | f"Result for Table {tableName}" >> beam.Map(print)
            )    
    
    p.run().wait_until_finish()

Processing table DimCities for 2013-01-01 00:00:00.
Processing table DimCustomers for 2013-01-01 00:00:00.
Processing table DimEmployees for 2013-01-01 00:00:00.
Processing table DimPaymentMethods for 2013-01-01 00:00:00.
Processing table DimStockItems for 2013-01-01 00:00:00.
Processing table DimSuppliers for 2013-01-01 00:00:00.
Processing table DimTransactionTypes for 2013-01-01 00:00:00.
Processing table DimDates for 2013-01-01 00:00:00.
{'Successfully processed table DimStockItems after 2012-12-31 00:00:00 to 2013-01-01 00:00:00.'}
{'Successfully processed table DimTransactionTypes after 2012-12-31 00:00:00 to 2013-01-01 00:00:00.'}
{'Successfully processed table DimDates after 2012-12-31 00:00:00 to 2013-01-01 00:00:00.'}
{'Successfully processed table DimSuppliers after 2012-12-31 00:00:00 to 2013-01-01 00:00:00.'}
{'Successfully processed table DimEmployees after 2012-12-31 00:00:00 to 2013-01-01 00:00:00.'}
{'Successfully processed table DimPaymentMethods after 2012-12-31 00:0